Part F — Technical Requirements
Question 38
Use Spark transformations wherever practical.

Include examples of:

select
withColumn
when
otherwise
cast
trim
upper/lower/initcap
regexp_replace
to_date
coalesce
join
groupBy
agg
countDistinct
sum
avg
row_number
rank or dense_rank
Window
Question 39
Avoid collecting the full source dataset to the driver.

Small reference sheets may be converted through Python when necessary, but the appointment transformations and aggregations must be performed using Spark.

Question 40
Make the pipeline rerunnable.

A second execution should not create uncontrolled duplicate data.

Document whether you use:

overwrite
append with deduplication
run-specific paths
Question 41
Add Markdown documentation before each major layer:

RAW
BRONZE
SILVER
GOLD
Explain:

Purpose
Input
Output
Main transformations
Validation performed
Question 42
At the end of the notebook, print or display a control summary:

Raw files copied
Bronze rows
Silver clean rows
Silver rejected rows
Gold tables created
Reconciliation passed
Pipeline status
Deliverables
Submit:

Databricks notebook exported as .dbc, .ipynb, or source file.
Screenshot of RAW folder.
Screenshot of Bronze outputs.
Screenshot of Silver clean and rejected counts.
Screenshot of Gold department report.
Screenshot of Gold doctor ranking.
Screenshot of the data-quality report.
A short document containing the ten business insights.
A brief explanation of the medallion architecture used.
The final reconciliation result.

Databricks Medallion Architecture Assignment
Owen Peterson
7/18/26
#TODO


In [0]:
import os

base_directory: str = os.path.dirname(__file__)
hospital_medallion_path: str = os.path.join(base_directory, "Volumes/tables/medallion_hospital")
input_path: str = os.path.join(hospital_medallion_path, "input")
raw_path: str = os.path.join(hospital_medallion_path, "raw")
bronze_path: str = os.path.join(hospital_medallion_path, "bronze")
silver_path: str = os.path.join(hospital_medallion_path, "silver")
gold_path: str = os.path.join(hospital_medallion_path, "gold")

# Raw
#TODO
Purpose Input Output Main transformations Validation

In [ ]:
#TODO copy input into raw
#TODO print files in raw and prove copied succesffuly


import shutil
import os

os.makedirs(raw_path, exist_ok=True)

for file_name in os.listdir(input_path):
    src: str = os.path.join(input_path, file_name)
    dst: str = os.path.join(raw_path, file_name)
    shutil.copy2(src, dst)

display([f for f in os.listdir(raw_path)])

In [ ]:
#TODO copy input into raw
#TODO print files in raw and prove copied succesffuly


import shutil
import os

os.makedirs(raw_path, exist_ok=True)

for file_name in os.listdir(input_path):
    src: str = os.path.join(input_path, file_name)
    dst: str = os.path.join(raw_path, file_name)
    shutil.copy2(src, dst)

display([f for f in os.listdir(raw_path)])

The raw layer should not have business transformations, it is the source data and should stay that way for auditability and repeatability of the pipeline

In [ ]:
from pyspark import SparkContext
from pyspark.sql import DataFrame, SparkSession, Column

spark: SparkSession = (SparkSession.builder.
                        appName("Hospital Medallion Pipeline")
                       .getOrCreate())
sc: SparkContext = spark.sparkContext

# Bronze
Purpose Input Output Main transformations Validation

In [0]:
#TODO Question 7
#Read hospital_appointments_raw.csv into a Spark DataFrame.

#Use options appropriate for a header-based CSV file.

#Initially load source columns in a way that prevents invalid values from being silently lost.
hospital_appointments_file_name: str = "hospital_appointments_raw.csv"
hospital_appointments_path: str = os.path.join(raw_path, hospital_appointments_file_name)
hospital_appointments_raw_df: DataFrame = spark.read.csv(
    hospital_appointments_path,
    header=True,
    inferSchema=False,
    mode="PERMISSIVE",)

In [0]:
#Read all Excel sheets required for the assignment.

#Create individual DataFrames for:

#doctor_master
#department_targets
#status_mapping
#data_dictionary
#Use a method supported by your workspace. A Python Excel library may be used to read the workbook and then convert each sheet to a Spark DataFrame.

import pandas as pd

hospital_reference_master_path: str = os.path.join(raw_path, "hospital_reference_master.xlsx")

doctor_master_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="doctor_master")
department_targets_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="department_targets")
status_mapping_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="status_mapping")
data_dictionary_pd: pd.DataFrame = pd.read_excel(hospital_reference_master_path, sheet_name="data_dictionary")

doctor_master_df: DataFrame = spark.createDataFrame(doctor_master_pd)
department_targets_df: DataFrame = spark.createDataFrame(department_targets_pd)
status_mapping_df: DataFrame = spark.createDataFrame(status_mapping_pd)
data_dictionary_df: DataFrame = spark.createDataFrame(data_dictionary_pd)

display({
    "reference_workbook": hospital_reference_master_path,
    "doctor_master_rows": doctor_master_df.count(),
    "department_targets_rows": department_targets_df.count(),
    "status_mapping_rows": status_mapping_df.count(),
    "data_dictionary_rows": data_dictionary_df.count(),
})


In [0]:
#Question 9
#Add the following audit columns to the appointment Bronze DataFrame:

#source_file_name
#source_system_name
#bronze_ingestion_timestamp
#bronze_ingestion_date
#record_hash
import pyspark.sql.functions as F

hospital_appointments_bronze_df: DataFrame = (hospital_appointments_raw_df
                                              .withColumn("source_file_name", F.lit(hospital_appointments_file_name))
                                              .withColumn("source_system_name", F.lit("Hospital Appointment and Revenue Analytics"))
                                              .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
                                              .withColumn("bronze_ingestion_date", F.current_date())
#TODO check to see if these hashes are sufficient or if it should be on the full column
                                              .withColumn("record_hash", F.hash(
                                                  "appointment_id",
                                                  "patient_id",
                                                  "doctor_id",
                                              )))


In [0]:
#Write the appointment Bronze DataFrame and all Excel reference DataFrames to the Bronze layer.

#Suggested names:

#bronze_appointments
#bronze_doctor_master
#bronze_department_targets
#bronze_status_mapping
#bronze_data_dictionary
#Use Delta format when supported. Otherwise, use Parquet and clearly document the choice.


hospital_appointments_bronze_path: str = os.path.join(bronze_path, "bronze_appointments")
(hospital_appointments_bronze_df.write
 .format("delta")
 .mode("overwrite")
 .save(hospital_appointments_bronze_path))

doctor_master_bronze_path: str = os.path.join(bronze_path, "doctor_master")
department_targets_bronze_path: str = os.path.join(bronze_path, "department_targets")
status_mapping_bronze_path: str = os.path.join(bronze_path, "status_mapping")
data_dictionary_bronze_path: str = os.path.join(bronze_path, "data_dictionary")

(doctor_master_df.write
 .format("delta")
 .mode("overwrite")
 .save(doctor_master_bronze_path))
(department_targets_df.write
 .format("delta")
 .mode("overwrite")
 .save(department_targets_bronze_path))
(status_mapping_df.write
 .format("delta")
 .mode("overwrite")
 .save(status_mapping_bronze_path))
(data_dictionary_df.write
.format("delta")
.mode("overwrite")
.save(data_dictionary_bronze_path))



In [0]:
#Show:

#Bronze row count
#Distinct appointment ID count
#Exact duplicate count using record_hash
#Duplicate business-key count using appointment_id


appointment_row_count: int = hospital_appointments_bronze_df.count()
distinct_appointment_id_count = (hospital_appointments_bronze_df
                                 .select("appointment_id")
                                 .distinct()
                                 .count())
exact_duplicate_count: int = (hospital_appointments_bronze_df
                         .groupBy("record_hash")
                         .count()
                         .where("count > 1")
                         .count())
duplicate_business_key_count: int = (hospital_appointments_bronze_df
                                .groupBy("appointment_id")
                                .count()
                                .where("count > 1")
                                .count())
print(f"Appointment row count: {appointment_row_count}")
print(f"Distinct appointment ID count: {distinct_appointment_id_count}")
print(f"Exact duplicate count: {exact_duplicate_count}")
print(f"Duplicate business-key count: {duplicate_business_key_count}")


%md
# Silver
Purpose Input Output Main transformations Validation

In [0]:
# Part D — Silver Data Quality and Transformation
# Perform the following transformations.
#
# Question 13 — Trim and standardize text
# Apply trimming and case standardization to:
#
# patient_name
# city
# department
# doctor_name
# appointment_status
# payment_mode
# source_system
# Expected examples:
#
# "  Diya Sharma  " → "Diya Sharma"
# "chennai" → "Chennai"
# "ORTHOPEDICS" → "Orthopedics"
# "dr. meera iyer" → "Dr. Meera Iyer"

def normalize(column_name: str) -> Column:
    return F.initcap(F.lower(F.trim(F.col(column_name))))


hospital_appointments_silver_df: DataFrame = (spark.read
                                              .format("delta")
                                              .load(hospital_appointments_bronze_path))
hospital_appointments_silver_df = (hospital_appointments_silver_df
                                   .withColumns({
    "patient_name": normalize("patient_name"),
    "city": normalize("city"),
    "department": normalize("department"),
    "doctor_name": normalize("doctor_name"),
    "appointment_status": normalize("appointment_status"),
    "payment_mode": normalize("payment_mode"),
    "source_system": normalize("source_system")
}))

In [0]:
# Question 14 — Standardize appointment status
# Join with status_mapping and derive:
#
# standard_appointment_status
# is_billable
# include_in_utilization
# Unmapped statuses must be marked as rejected or assigned a clear validation failure.




In [0]:
Question 15 — Convert and validate appointment dates
Convert appointment_date to a valid date.

Support the formats present in the source data.

Create:

appointment_date_clean
appointment_year
appointment_month
appointment_day
Invalid dates must not be accepted as valid.

In [0]:
Question 16 — Convert numeric columns
Convert these columns safely:

age
consultation_fee
discount_pct
amount_paid
Rows that cannot be converted must be flagged.

In [0]:
Question 17 — Validate age
Valid age range:

0 to 110
Flag negative ages and unrealistic ages.



In [0]:
Question 18 — Validate gender
Accepted values:

M
F
O
Any other value must be rejected or marked invalid.

In [0]:
Question 19 — Standardize department values
Map known aliases where appropriate.

Example:

Cardio → Cardiology
After standardization, the department must exist in department_targets.

In [0]:
Question 20 — Validate doctor details
Join the appointment data with doctor_master.

Validate that:

doctor_id exists
Doctor is active
Doctor name matches the master
Doctor belongs to the stated department
Use the master doctor name and department in the clean output.

In [0]:
Question 21 — Handle duplicate records
Create rules for:

Exact duplicate records
Duplicate appointment_id values
For duplicate appointment IDs, keep one deterministic record and reject the remaining records. Document the ordering rule used.

In [0]:
Question 22 — Handle null values
Apply appropriate rules:

Missing patient ID → reject
Missing patient name → reject
Null discount percentage → treat as 0
Blank payment mode → allowed only when amount paid is 0
Missing required master-data match → reject

In [0]:
Question 23 — Validate consultation fee
Rules:

Must be numeric
Must be greater than or equal to zero

In [0]:
Question 24 — Validate discount
Rules:

0 <= discount_pct <= 100

In [0]:
Question 25 — Calculate expected amount
Create:

expected_amount_paid
Suggested formula for completed/billable appointments:

consultation_fee × (1 - discount_pct / 100)
For non-billable statuses, expected paid amount should normally be zero.

In [0]:
Question 26 — Validate amount paid
Flag:

Negative amount paid
Amount paid much higher than the expected amount
Completed appointments where actual amount differs from expected amount
Non-billable appointments with a non-zero payment
Use a small tolerance for decimal comparison.

In [0]:
Question 27 — Validate payment mode
Accepted paid transaction modes:

UPI
CARD
CASH
NET_BANKING
Blank is allowed only when no amount was paid.

In [0]:
Question 28 — Validate phone number
A valid phone number must contain exactly 10 numeric digits.

Create a masked phone field for the Silver layer, for example:

98******21
Do not expose the full phone number in Gold reports.

In [0]:
Question 29 — Create validation columns
Create at least:

is_valid_record
validation_error_count
validation_errors
silver_processed_timestamp
validation_errors should contain one or more readable error reasons.

Example:

INVALID_AGE|INVALID_PHONE

In [0]:
Question 30 — Split valid and rejected data
Create:

silver_appointments_clean
silver_appointments_rejected
The rejected dataset must retain:

Original identifying columns
Validation reasons
Source file
Bronze ingestion timestamp
Silver processing timestamp

In [0]:
Question 31 — Silver output
Write both clean and rejected datasets into the Silver layer.

Display:

Bronze row count
Clean Silver row count
Rejected Silver row count
Reconciliation result
Required reconciliation:

Bronze count = Clean Silver count + Rejecte

# Gold
Purpose Input Output Main transformations Validation

In [0]:
Part E — Gold Layer: Report-Ready Outputs
Create the following Gold outputs from clean Silver data.

Question 32 — Department monthly performance
Create a monthly department-level report with:

appointment_year
appointment_month
department
total_appointments
completed_appointments
cancelled_appointments
no_show_appointments
scheduled_appointments
completion_rate_pct
cancellation_rate_pct
no_show_rate_pct
gross_consultation_value
discount_value
net_revenue
average_revenue_per_completed_appointment
unique_patients
monthly_completed_target
monthly_revenue_target
completed_target_achievement_pct
revenue_target_achievement_pct
target_status
Join the report with department_targets.

Suggested target status:

ACHIEVED
PARTIALLY_ACHIEVED
NOT_ACHIEVED

In [0]:
Question 33 — Doctor performance report
Create a doctor-level report with:

doctor_id
doctor_name
department
specialization
total_appointments
completed_appointments
cancelled_appointments
no_show_appointments
completion_rate_pct
no_show_rate_pct
unique_patients
net_revenue
average_revenue_per_completed_appointment
Rank doctors within each department by:

Net revenue
Completed appointments

In [0]:
Question 34 — Daily operational trend
Create a daily report:

appointment_date_clean
department
total_appointments
completed_appointments
cancelled_appointments
no_show_appointments
net_revenue

In [0]:
Question 35 — Source-system performance
Create a source-system report:

source_system
total_appointments
completed_appointments
conversion_to_completed_pct
cancelled_appointments
no_show_appointments
net_revenue
average_revenue

In [0]:
Question 36 — Data-quality summary
Create a Gold quality report containing:

quality_rule
failed_record_count
failed_record_pct
Include at least these quality categories:

Duplicate records
Invalid date
Missing patient
Invalid age
Invalid gender
Invalid department
Invalid doctor
Invalid fee
Invalid discount
Invalid amount paid
Invalid payment mode
Invalid phone
Invalid status

In [0]:
Question 37 — Top business insights
Using the Gold outputs, answer these questions:

Which department generated the highest net revenue?
Which department had the highest cancellation rate?
Which department had the highest no-show rate?
Which doctor completed the most appointments?
Which doctor generated the highest revenue?
Which source system produced the highest completed conversion rate?
Which month generated the highest revenue?
Which department missed its monthly target by the largest percentage?
What are the three most common data-quality failures?
How many records were rejected from the source?